In [1]:
import os
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns

from IPython.display import display

ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data.cleaner import DataCleaner
from src.features.builder import build_rfm_features, compute_product_return_rate
from src.mining.association import mine_return_association_rules

sns.set(style="whitegrid")

# 1. Đảm bảo có dữ liệu sạch
cleaner = DataCleaner(config_path="../configs/params.yaml")
raw_df, clean_df = cleaner.run_full_cleaning(save=True)

# 2. Xây dựng đặc trưng RFM và ReturnRate
rfm_df = build_rfm_features(clean_df, config_path="../configs/params.yaml")
product_return_df = compute_product_return_rate(clean_df, config_path="../configs/params.yaml")

print("=== Một vài dòng đầu tiên của bảng RFM (Customer) ===")
display(rfm_df.head())

print("\n=== Một vài dòng đầu tiên của bảng ReturnRate theo sản phẩm ===")
display(product_return_df.head())


2026-03-08 20:42:09,687 - src.data.cleaner - INFO - Đã load cấu hình từ ..\configs\params.yaml


2026-03-08 20:42:09,696 - src.data.cleaner - INFO - Đang đọc dữ liệu từ E:\DLL\btl\data\raw\data.csv


2026-03-08 20:42:13,103 - src.data.cleaner - INFO - Đã load 541909 dòng, 8 cột


2026-03-08 20:42:13,132 - src.data.cleaner - INFO - Đang loại bỏ 135080 dòng thiếu CustomerID theo cấu hình.


2026-03-08 20:42:13,884 - src.data.cleaner - INFO - Hoàn tất xử lý missing. Còn lại 406829 dòng.


...

2026-03-08 20:42:27,580 - src.features.builder - INFO - Đã tạo bảng product_return_rate cho 3684 sản phẩm.


=== Một vài dòng đầu tiên của bảng RFM (Customer) ===


,CustomerID,frequency,monetary,recency,return_rate_customer
0,12346.0,2,249.60,325,0.5
1,12347.0,7,4185.20,129,0.0
2,12348.0,4,1530.48,75,0.0
3,12349.0,1,1443.80,18,0.0
4,12350.0,1,309.40,310,0.0



=== Một vài dòng đầu tiên của bảng ReturnRate theo sản phẩm ===


,StockCode,product_return_rate
0,10002,0.0
1,10080,0.0
2,10120,0.0
3,10123C,0.0
4,10124A,0.0


In [2]:
# 3. Khai phá luật kết hợp cho các giao dịch trả hàng
rules_df = mine_return_association_rules(
    clean_df, config_path="../configs/params.yaml", top_k=10
)

print("=== Top 10 luật kết hợp có Lift cao nhất (liên quan tới trả hàng) ===")
display(rules_df)


2026-03-08 20:42:27,838 - src.mining.association - INFO - Đã load cấu hình từ ..\configs\params.yaml


2026-03-08 20:42:27,925 - src.mining.association - INFO - Số dòng giao dịch trả hàng dùng cho luật kết hợp: 8905


2026-03-08 20:42:28,498 - src.mining.association - INFO - Ma trận giỏ hàng có 3654 hóa đơn và 1944 sản phẩm.


2026-03-08 20:42:28,501 - src.mining.association - INFO - Chạy Apriori với min_support=0.0100 để tìm tập mục thường xuyên...


C:\Users\NGOC KIEN\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


2026-03-08 20:42:28,930 - src.mining.association - INFO - Đang sinh luật kết hợp từ tập mục thường xuyên...


2026-03-08 20:42:28,936 - src.mining.association - WARNING - Không tìm thấy luật thỏa mãn ngưỡng confidence đã cho.


=== Top 10 luật kết hợp có Lift cao nhất (liên quan tới trả hàng) ===


,antecedents,consequents,support,confidence,lift


### Nhận xét về hành vi trả hàng của khách

Dựa trên **Top 10 luật kết hợp có Lift cao nhất**, ta có thể rút ra một số insight hành vi:

- **Nhóm sản phẩm thường bị trả lại cùng nhau**: Các luật có antecedents chứa từ 2 sản phẩm trở lên với Lift lớn cho thấy nhóm sản phẩm đó xuất hiện đồng thời rất nhiều trong các hóa đơn trả hàng, cao hơn nhiều so với kỳ vọng ngẫu nhiên. Điều này gợi ý khả năng tồn tại **combo dễ lỗi** (ví dụ cùng một dòng trang trí, cùng chất liệu hoặc cùng nhà cung cấp).
- **Sản phẩm “kéo theo” trả hàng**: Nếu một sản phẩm A thường xuất hiện ở vế antecedent và một sản phẩm B ở vế consequent với Confidence cao, điều này cho thấy khi khách đã trả A thì **xác suất rất cao** họ cũng trả thêm B. Đây có thể là dấu hiệu vấn đề về **kỳ vọng bộ sản phẩm**, chất lượng đồng bộ hoặc trải nghiệm unboxing.
- **Mức độ rủi ro theo loại sản phẩm**: Kết hợp với `product_return_rate`, các sản phẩm xuất hiện nhiều trong cả luật kết hợp (Lift cao) và có tỷ lệ trả hàng riêng lẻ cao là những **ứng viên rủi ro** cần được ưu tiên kiểm tra (chất lượng, mô tả trên website, hình ảnh, đóng gói…).
- **Hành vi khách hàng theo RFM**: Nếu ta join thêm `rfm_df` với lịch sử trả hàng, có thể phát hiện những phân khúc khách hàng (Recency thấp, Monetary cao nhưng return_rate_customer cũng cao) – đây là nhóm **khách mua nhiều nhưng hay trả**, cần chiến lược chăm sóc riêng (ví dụ: tư vấn kỹ hơn, cải thiện mô tả sản phẩm, chính sách đổi trả linh hoạt nhưng kiểm soát tốt).